### Experiment: Effect of Prediction Head Capacity

In this experiment we study how the capacity of the prediction head affects performance when the temporal context is limited.

We use the same **nonlinear sequence simulation** described earlier. The sequence contains community-structured transitions where the traversal direction depends on the **parity of the sum of the previous \(K\) community visits**, introducing a long-range dependency.

The recurrent backbone is trained with a **fixed BPTT length of 3**, which is intentionally shorter than the dependency length. As a result, the recurrent layer alone cannot fully capture the required temporal structure.

To investigate whether downstream model capacity can compensate for this limitation, we gradually increase the **number of neurons in the prediction head** (MLP head) while keeping all other components fixed.

#### Experimental Setup

- Dataset: nonlinear community traversal sequence  
- Recurrent backbone: RNN  
- BPTT length: **3**  
- Training regime: single-pass online learning  
- Prediction head: MLP with varying hidden size  

#### Variable

We vary the **number of neurons in the prediction head**:


In [1]:
## Load necessary library files ##

import sys
sys.path.append('..')
from source.utils import get_sequence, DatasetConverter
from source.model.model import Model

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch import from_numpy as tnsr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
import pickle 

In [2]:
## select device ##
device = torch.device("cpu")

print("Using device:", device)

Using device: cpu


In [3]:
## load the nonlinear simulation from source files ##
print("A 42 tokens long nonlinear sequence ", get_sequence(42, n_community=2, n_members=3, context_depth=4))

A 42 tokens long nonlinear sequence  ABCGDEFGDEFGABCGDEFGDFEGDFEGDFEGDEFGDEFGDE


In [4]:
total_samples, n_community, n_members, context_depth = 100000, 2, 3, 4
total_layers, head_layers, short_term_memory = 2, 1, 4

vocab_size = n_community*n_members + 1


In [8]:
reps = 10
neurons_per_head = [10]
res = []
repititions = []
head_size = []
samples_seen = []

for rep in tqdm(range(reps)):
    for head_neuron in neurons_per_head:
        model = Model(
        total_layers = total_layers,
        num_layers_prediction_head = head_layers,

        # ---- Layer sizes ----
        vocab_size = vocab_size,                  # layer 0 input dimension
        hidden_sizes = [50, 50],    # H0, H1
        prediction_sizes = [head_neuron]*total_layers,
        embedding_dim = 30,

        # ---- Learning rates per layer ----
        lr_layers = 1e-4,   

        # ---- Optimizer type (user can choose) ----
        optimizer_class = torch.optim.Adam,
        optimizer_kwargs = {
            "weight_decay": 1e-12
        },

        # ---- Sleep hyperparameters ----
        short_term_memory = short_term_memory,
        context_tag_buffer_size=20,
        # ---- Misc ----
        recon_threshold = 1e-3,
        device = device
    )

    data = get_sequence(total_samples, n_community, n_members, context_depth=context_depth, train_percent=1.0)
    dataset = DatasetConverter(data, short_term_memory=short_term_memory)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    
    ii = 0 
    h_ = None
    correct_ring = np.zeros(1000)
    for x, y in loader:
        x = x.to(device).long()
        y = y.to(device).long()
        logits, loss, recon_loss, h_ = model.wake_step(x, y, h_)


        with torch.no_grad():
            ii += 1
            pred_tok = logits.argmax(dim=-1)
            correct_ring[ii % 1000] = (pred_tok[0] == y[0, 0]).item()
            
            if ii%1000 == 0:
                acc = np.sum(correct_ring) / (1000 if ii >= 1000 else ii)
                res.append(acc)
                samples_seen.append(ii)
                repititions.append(rep)
                head_size.append(head_neuron)

                print("Iter ", ii, f"prediction loss: {loss:.8e}", f"Memory loss: {recon_loss:.8e}", "Acc: ", acc)
                if model.sleeping:
                    print("Sleep on ", model.recon_loss_ema)

        if ii%20000==0:
            model.sleep(total_steps=1000)


df = pd.DataFrame()
df['reps'] = repititions
df['samples seen'] = samples_seen
df['head size'] = head_size
df['Accuracy'] = res

with open('../pickle_files/head_complexity.pickle', 'wb') as f:
    pickle.dump(df, f)

  0%|          | 0/10 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x50 and 10x10)

In [7]:
model


Model(
  (memories): ModuleList(
    (0): Memory(
      (embedding): Embedding(7, 30)
      (encoder): RNN(30, 50, batch_first=True)
      (decoder): RNN(7, 50, batch_first=True)
      (out): Linear(in_features=50, out_features=7, bias=True)
    )
    (1): Memory(
      (embedding): Linear(in_features=50, out_features=30, bias=True)
      (encoder): RNN(30, 50, batch_first=True)
      (decoder): RNN(50, 50, batch_first=True)
      (out): Linear(in_features=50, out_features=50, bias=True)
    )
  )
  (heads): ModuleList(
    (0): PredictionFiLM(
      (layers): ModuleList(
        (0): Linear(in_features=10, out_features=10, bias=True)
      )
      (film): Sequential(
        (0): Linear(in_features=50, out_features=20, bias=True)
      )
    )
    (1): PredictionFiLM(
      (layers): ModuleList(
        (0): Linear(in_features=10, out_features=10, bias=True)
      )
    )
  )
)